# Lab | Error Handling

Objective: Practice how to identify, handle and recover from potential errors in Python code using try-except blocks.

## Challenge 

Paste here your lab *functions* solutions. Apply error handling techniques to each function using try-except blocks. 

The try-except block in Python is designed to handle exceptions and provide a fallback mechanism when code encounters errors. By enclosing the code that could potentially throw errors in a try block, followed by specific or general exception handling in the except block, we can gracefully recover from errors and continue program execution.

However, there may be cases where an input may not produce an immediate error, but still needs to be addressed. In such situations, it can be useful to explicitly raise an error using the "raise" keyword, either to draw attention to the issue or handle it elsewhere in the program.

Modify the code to handle possible errors in Python, it is recommended to use `try-except-else-finally` blocks, incorporate the `raise` keyword where necessary, and print meaningful error messages to alert users of any issues that may occur during program execution.



In [1]:
from numbers import Real

def initialize_inventory(products):
    """Collect non-negative integer quantities; retry invalid entries."""
    try:
        if not isinstance(products, (list, tuple)) or not all(
                isinstance(product, str) and product.strip() for product in products):
            raise TypeError("Products must be a list or tuple of non-empty names")
        if len(set(products)) != len(products):
            raise ValueError("Product names must be unique")
    except (TypeError, ValueError) as error:
        print(f"Inventory error: {error}")
        return None
    else:
        inventory = {}
        for product in products:
            while True:
                try:
                    quantity = int(input(f"Enter the quantity for {product}: "))
                    if quantity < 0:
                        raise ValueError("Quantity cannot be negative")
                except ValueError as error:
                    print(f"Invalid quantity: {error}. Try again.")
                except (EOFError, KeyboardInterrupt):
                    print("Inventory initialization cancelled.")
                    return None
                else:
                    inventory[product] = quantity
                    break
        return inventory

def get_customer_orders():
    """Collect unique non-empty product names, validating yes/no answers."""
    customer_orders = set()
    try:
        while True:
            try:
                product = input("Enter a product name: ").strip()
                if not product:
                    raise ValueError("Product name cannot be empty")
            except ValueError as error:
                print(f"Invalid order: {error}")
                continue
            else:
                customer_orders.add(product)
            while True:
                try:
                    another = input("Add another product? (yes/no): ").strip().lower()
                    if another not in {"yes", "no"}:
                        raise ValueError("Answer yes or no")
                except ValueError as error:
                    print(f"Invalid answer: {error}")
                else:
                    break
            if another == "no":
                return customer_orders
    except (EOFError, KeyboardInterrupt):
        print("Order collection cancelled.")
        return None

def update_inventory(customer_orders, inventory):
    """Validate the entire order before changing inventory, preventing partial updates."""
    try:
        if not isinstance(customer_orders, set) or not isinstance(inventory, dict):
            raise TypeError("Expected a set of orders and an inventory dictionary")
        for product in customer_orders:
            if product not in inventory:
                raise KeyError(f"Unknown product: {product}")
            quantity = inventory[product]
            if not isinstance(quantity, int) or isinstance(quantity, bool):
                raise TypeError(f"Invalid quantity for {product}")
            if quantity <= 0:
                raise ValueError(f"Out of stock: {product}")
    except (TypeError, ValueError, KeyError) as error:
        print(f"Cannot update inventory: {error}")
        return None
    else:
        for product in customer_orders:
            inventory[product] -= 1
        return inventory

def calculate_order_statistics(customer_orders, products):
    try:
        if not isinstance(customer_orders, set) or not isinstance(products, (list, tuple)):
            raise TypeError("Expected an order set and a product list")
        if not all(isinstance(product, str) and product for product in products):
            raise ValueError("Product names must be non-empty strings")
        if len(set(products)) != len(products):
            raise ValueError("Product names must be unique")
        if not customer_orders.issubset(set(products)):
            raise ValueError("Orders contain unknown products")
        total = len(customer_orders)
        percentage = total / len(products) * 100
    except (TypeError, ValueError, ZeroDivisionError) as error:
        print(f"Statistics error: {error}")
        return None
    else:
        return total, percentage

def print_order_statistics(order_statistics):
    try:
        if not isinstance(order_statistics, (tuple, list)) or len(order_statistics) != 2:
            raise ValueError("Statistics must contain total and percentage")
        total, percentage = order_statistics
        if not isinstance(total, int) or isinstance(total, bool) or total < 0:
            raise ValueError("Total must be a non-negative integer")
        if not isinstance(percentage, Real) or isinstance(percentage, bool) or not 0 <= percentage <= 100:
            raise ValueError("Percentage must be a number between 0 and 100")
    except (TypeError, ValueError) as error:
        print(f"Cannot print statistics: {error}")
        return False
    else:
        print("Order Statistics:")
        print(f"Total Products Ordered: {total}")
        print(f"Percentage of Unique Products Ordered: {percentage}%")
        return True

def print_updated_inventory(inventory):
    try:
        if not isinstance(inventory, dict):
            raise TypeError("Inventory must be a dictionary")
        for product, quantity in inventory.items():
            if not isinstance(product, str) or not product:
                raise ValueError("Product names must be non-empty strings")
            if not isinstance(quantity, int) or isinstance(quantity, bool) or quantity < 0:
                raise ValueError(f"Invalid quantity for {product}")
    except (TypeError, ValueError) as error:
        print(f"Cannot print inventory: {error}")
        return False
    else:
        print("Updated Inventory:")
        for product, quantity in inventory.items():
            print(f"{product}: {quantity}")
        return True

def manage_orders(products):
    """Run the original workflow, stopping cleanly when a stage fails."""
    try:
        inventory = initialize_inventory(products)
        if inventory is None:
            return None
        orders = get_customer_orders()
        if orders is None:
            return None
        updated = update_inventory(orders, inventory)
        if updated is None:
            return None
        statistics = calculate_order_statistics(orders, products)
        if statistics is None:
            return None
        print_order_statistics(statistics)
        print_updated_inventory(updated)
        return updated, statistics
    finally:
        print("Order management session finished.")

products = ["t-shirt", "mug", "hat", "book", "keychain"]
result = manage_orders(products)

Invalid quantity: invalid literal for int() with base 10: 'abc'. Try again.
Invalid quantity: Quantity cannot be negative. Try again.
Invalid order: Product name cannot be empty
Invalid answer: Answer yes or no
Order Statistics:
Total Products Ordered: 1
Percentage of Unique Products Ordered: 20.0%
Updated Inventory:
t-shirt: 3
mug: 1
hat: 1
book: 0
keychain: 5
Order management session finished.


Based on the six customer-order functions in jordicallejo/lab-python-functions. Invalid input is retried or reported with a None/False result; the workflow checks those results before continuing. Inventory updates are atomic: unknown or out-of-stock products leave the entire inventory unchanged. The finally block reports session completion on success, error or cancellation.